# v23 — storey_pct + comp_6m_count (comp medians removed)

Testing the non-redundant comp features only:
- storey_pct — mid_storey / max_floor_lvl, relative floor position [0-1]
- comp_6m_count — # transactions same (town, flat_model) in past 6 months (liquidity signal)

comp_6m_median / comp_12m_median / comp_floor_adj_median removed from pipeline
because they were collinear with existing OOF price encodings (yr_median_price,
flat_model_median_price), causing $225 OOF degradation in v22.

Baseline to beat: v20 OOF $21,465 (same model params, no comp features).

In [1]:
# ── EDIT ONLY THIS CELL ────────────────────────────────────────────────────
CONFIG = {
    "experiment_name": "v23",

    "data": {
        "train_path": "../../data/raw/train.csv",
        "test_path":  "../../data/raw/test.csv",
    },

    "features": {
        "encode_pairs": [
            ("town",           "town_median_price",           1),
            ("postal_sector",  "postal_sector_median_price",  1),
            ("flat_model",     "flat_model_median_price",     1),
            ("town_flat_type", "town_flat_type_median_price", 3),
            ("Tranc_Year",     "year_median_price",           1),
        ],
        "drop_cols": [],
    },

    "models": {
        "active": ["xgb", "lgbm", "et", "catboost"],

        # XGB — same as v20
        "xgb": {
            "n_estimators": 2000, "max_depth": 7, "learning_rate": 0.05,
            "subsample": 0.9, "colsample_bytree": 0.4, "min_child_weight": 7,
            "reg_alpha": 0.01, "reg_lambda": 1.5,
            "random_state": 42, "n_jobs": -1, "verbosity": 0, "tree_method": "hist",
        },

        # LGBM — Optuna TPE 100-trial best (from v20)
        "lgbm": {
            "n_estimators": 888, "max_depth": 14, "num_leaves": 317,
            "learning_rate": 0.0272, "subsample": 0.9225, "colsample_bytree": 0.4564,
            "min_child_samples": 31, "reg_alpha": 0, "reg_lambda": 1.135,
            "random_state": 42, "n_jobs": -1, "verbosity": -1,
        },

        # ET — Optuna 50-trial best (from v20)
        "et": {
            "n_estimators": 600, "min_samples_split": 7, "min_samples_leaf": 2,
            "max_features": 0.829, "max_depth": 24,
            "random_state": 42, "n_jobs": -1,
        },

        # CatBoost — Optuna TPE 100-trial best (from v20)
        "catboost": {
            "iterations": 1994, "depth": 10,
            "learning_rate": 0.0585,
            "l2_leaf_reg": 1.4962, "colsample_bylevel": 0.7696,
            "min_child_samples": 20,
            "loss_function": "RMSE", "random_seed": 42, "verbose": 0,
        },

        "ridge_alphas": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    },

    "cv": {"n_splits": 5, "random_state": 42},

    "output": {
        "submission_path": "../../outputs/submissions/sub_v23.csv",
        # "fulldata_submission_path": "../../outputs/submissions/sub_v23_fulldata.csv",
        "sample_path": "../../outputs/submissions/sample_sub_reg.csv",
    },
}


In [2]:
import sys
sys.path.insert(0, "../../")

from src.pipeline import run_pipeline

results = run_pipeline(CONFIG)

/Users/tehmenghai/miniconda3/envs/ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/05/01 12:56:53 INFO mlflow.tracking.fluent: Experiment with name 'v23' does not exist. Creating a new experiment.



Experiment: v23
Loaded — Train: (150634, 77)  |  Test: (16737, 76)
After engineering — Train: (150634, 101)  |  Test: (16737, 100)
After comp features — Train: (150634, 102)  |  Test: (16737, 101)
OOF encodings applied: ['town_median_price', 'postal_sector_median_price', 'flat_model_median_price', 'town_flat_type_median_price', 'year_median_price']
Feature matrix: 80 columns

Running 5-fold stacking...
 Fold  XGB-RMSE        LGBM-RMSE       ET-RMSE         CATBOOST-RMSE 
-----------------------------------------------------------------------
    1  $       21,540  $       21,474  $       22,894  $       21,511
    2  $       22,338  $       22,286  $       23,569  $       22,265
    3  $       21,675  $       21,628  $       23,118  $       21,733
    4  $       21,878  $       21,893  $       23,374  $       21,853
    5  $       21,853  $       21,693  $       23,194  $       21,725
-----------------------------------------------------------------------
 Mean  $       21,857  $     

In [3]:
print(f'OOF RMSE:      ${results["oof_rmse"]:,.0f}')
print(f'Ridge weights: {results["weights"]}')
print(f'Best alpha:    {results["best_alpha"]}')
print()
print('Per-model mean OOF RMSE:')
import numpy as np
for m, scores in results['fold_scores'].items():
    print(f'  {m:<12} ${np.mean(scores):>10,.0f}')
print()
print('To view all experiments in MLflow UI:')
print('  cd <project_root>')
print('  mlflow ui --backend-store-uri sqlite:///mlflow.db')
print('  Open http://localhost:5000')

OOF RMSE:      $21,500
Ridge weights: {'xgb': 0.21881340448683836, 'lgbm': 0.27105798080433535, 'et': 0.14159000575950736, 'catboost': 0.3709394139020219}
Best alpha:    0.001

Per-model mean OOF RMSE:
  xgb          $    21,857
  lgbm         $    21,795
  et           $    23,230
  catboost     $    21,817

To view all experiments in MLflow UI:
  cd <project_root>
  mlflow ui --backend-store-uri sqlite:///mlflow.db
  Open http://localhost:5000
